In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'SimHei'  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False  # 确保负号字符可用
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_boston
import seaborn as sns

# 线性回归-0416

## 手动梯度下降

特征数据先标准化处理，对梯度下降非常有帮助.

**注意：损失函数mse值在最佳拟合曲线位置本来就不会为0，这是正常的，我们要得到的是mse最小的那条拟合曲线。**

手动对比正规方程计算的mse：

((X @ w -y) ** 2).mean()

In [ ]:
boston = load_boston()
data = boston.data
target = boston.target
feature_names = boston.feature_names
df = pd.DataFrame(data=boston.data, columns=boston.feature_names)
df

In [ ]:
# 查看特征和目标的散点，并画出线性回归线。
rows=4
cols=4
fit, axes = plt.subplots(rows, cols, figsize=(6*cols, 4*rows))
axes = axes.flatten()
for i, col in enumerate(boston.feature_names):
    # sns.regplot(data=df, x=col, y=boston.target, ax=axes[i], ci=None, line_kws={'color':'red'})
    sns.regplot(x=df[col], y=boston.target, ax=axes[i], ci=None, line_kws={'color':'red'})
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

In [ ]:
# 特征相关性热力图
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')

In [ ]:
# 多特征分布图
df_t = df.copy()['target'] = boston.target
g = sns.pairplot(df_t)
g.savefig('特征分布图.png', dpi=300, bbox_inches='tight')

## 标准化

标准化的原理：

标准化一般的适用场景：适合手动梯度下降时时使用。

标准化后数据的前后对比：

In [ ]:
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(data), columns=feature_names)
n_feat = len(feature_names)
fig, axes = plt.subplots(n_feat, 2, figsize=(5*2, 5 * n_feat))
for i, col in enumerate(feature_names):
    sns.histplot(df[col], kde=True, ax=axes[i, 0], color='b')
    axes[i, 0].set_title(f'{col} 原始')
    sns.histplot(df_scaled[col], kde=True, ax=axes[i, 1], color='b')
    axes[i, 1].set_title(f'{col} 标准化后')


**可以看到，标准化处理后，无论原始数据量级是多少，全部缩放到均值0附近。**

## 归一化

归一化的原理：

归一化的适用场景：

归一化前后的数据对比

In [ ]:
scaler = MinMaxScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(data), columns=feature_names)
n_feat = len(feature_names)
fig, axes = plt.subplots(n_feat, 2, figsize=(5*2, 5 * n_feat))
for i, col in enumerate(feature_names):
    sns.histplot(df[col], kde=True, ax=axes[i, 0], color='b')
    axes[i, 0].set_title(f'{col} 原始')
    sns.histplot(df_scaled[col], kde=True, ax=axes[i, 1], color='b')
    axes[i, 1].set_title(f'{col} 归一化后')

标准化、归一化处理前后的数据分布图放一起对比

In [ ]:
scalerMinMax = MinMaxScaler()
scalerStand = StandardScaler()
df_MinMaxed = pd.DataFrame(scalerMinMax.fit_transform(data), columns=feature_names)
df_Standed = pd.DataFrame(scalerStand.fit_transform(data), columns=feature_names)
n_feat = len(feature_names)
fig, axes = plt.subplots(n_feat, 3, figsize=(5*3, 5 * n_feat))
for i, col in enumerate(feature_names):
    sns.histplot(df[col], kde=True, ax=axes[i, 0], color='b')
    axes[i, 0].set_title(f'{col} 原始')
    sns.histplot(df_MinMaxed[col], kde=True, ax=axes[i, 1], color='b')
    axes[i, 1].set_title(f'{col} 归一化后')
    sns.histplot(df_Standed[col], kde=True, ax=axes[i, 2], color='b')
    axes[i, 2].set_title(f'{col} 标准化后')

**想稳定训练，优先标准化。标准化的好处是每个特征数据波动1单位，代表的都是一倍的标准差。**

**想限制在固定区间（图片rgb，神经网络等），用归一化。**

## 手动计算

In [ ]:
# 归一化
min_val = data.min(axis=0)
max_val = data.max(axis=0)
x_norm = (data - min_val) / (max_val - min_val)

# 标准化
mean_val = data.mean(axis=0)
std_val = data.std(axis=0, ddof=0)
x_std = (data - mean_val) / std_val

In [ ]:
df_MinMaxed = pd.DataFrame(x_norm, columns=feature_names)
df_Standed = pd.DataFrame(x_std, columns=feature_names)
n_feat = len(feature_names)
fig, axes = plt.subplots(n_feat, 3, figsize=(5*3, 5 * n_feat))
for i, col in enumerate(feature_names):
    sns.histplot(df[col], kde=True, ax=axes[i, 0], color='b')
    axes[i, 0].set_title(f'{col} 原始')
    sns.histplot(df_MinMaxed[col], kde=True, ax=axes[i, 1], color='b')
    axes[i, 1].set_title(f'{col} 归一化后')
    sns.histplot(df_Standed[col], kde=True, ax=axes[i, 2], color='b')
    axes[i, 2].set_title(f'{col} 标准化后')

In [ ]:
# 对比sklearn和手动计算的归一化数据和标准差数据
np.allclose(x_norm, MinMaxScaler().fit_transform(data))

In [ ]:
np.allclose(x_std, StandardScaler().fit_transform(data))